# **DFE blocks without a timely symbolic solution: `dfe_timeout`**

To compute $\mathcal{R}_0$, the Jacobians $F$ and $V$ must be evaluated at the disease-free equilibrium (DFE). `pyR0compute` solves the DFE **block by block**: it splits the equations of the uninfected compartments into strongly connected components and solves them in topological order (see the notebook `03_nonlinear_systems.ipynb`).

Some coupled blocks have no reasonable symbolic solution, even though they can be solved numerically. In that case `sympy.solve` can keep computing for hours, and **it cannot be interrupted from within the same Python process**.

Since version 0.2.1, each hard block is solved in a **separate process with a time limit**, `dfe_timeout` (20 s by default):

* If SymPy finds the solution in time, the block is written in closed form, as before.
* Otherwise the process is stopped and the block stays **implicit**: its values are symbols $X^*$ defined by $\dot X = 0$. $\mathcal{R}_0$ is returned in compact form in terms of these $X^*$, and `dfe_numeric`, `R0_numeric`, `dfe_stability`, `prcc` and `sobol_indices` compute them numerically.
* Easy blocks (one equation of degree $\le 2$) are solved directly, at no extra cost.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/04_dfe_time_limit.ipynb)

In [1]:
%pip install pyR0compute                      # Install pyR0compute from PyPI

In [2]:
import time                                   # Measure computation times
import warnings                               # Show the warnings of the library
import numpy as np                            # Numerical computation
import sympy as sp                            # Symbolic computation
from pyr0compute import R0Model               # Main class of the library
from IPython.display import Markdown, display # Show equations in LaTeX

## 1. Within-host model with an immune regulation loop

Target cells $T$, infected cells $I$ and virus $V$, with three mediators of the innate response that form a **loop**: interferon $F$ activates NK cells $N$ (Hill function of order 2), NK cells activate macrophages $M$ and macrophages produce interferon. NK cells remove infected cells and macrophages remove virus.

$$
\begin{cases}
\dfrac{dT}{dt} = s - d_T T - \beta T V \\[6pt]
\dfrac{dI}{dt} = \beta T V - (\delta + k_N N)\, I \\[6pt]
\dfrac{dV}{dt} = p I - (c + k_M M)\, V \\[6pt]
\dfrac{dN}{dt} = a_N - d_N N + h_N \dfrac{F^2}{F^2 + K_F^2} \\[8pt]
\dfrac{dM}{dt} = a_M - d_M M + h_M \dfrac{N}{N + K_N} \\[8pt]
\dfrac{dF}{dt} = a_F - d_F F + h_F \dfrac{M}{M + K_M}
\end{cases}
$$

With $I = V = 0$, the DFE has two blocks: $[T]$, with $T^* = s/d_T$, and the loop $[N, M, F]$. After eliminating $M$ and $F$, the loop reduces to a cubic in $N$ with 121 terms in 12 parameters. `sympy.solve` on the coupled system does not finish: in version 0.2.0 this model was still computing after several minutes.

In [3]:
loop = '''
    dT/dt = s - d_T*T - beta*T*V
    dI/dt = beta*T*V - (delta + k_N*N)*I
    dV/dt = p*I - (c + k_M*M)*V
    dN/dt = a_N - d_N*N + h_N*F^2/(F^2 + K_F^2)
    dM/dt = a_M - d_M*M + h_M*N/(N + K_N)
    dF/dt = a_F - d_F*F + h_F*M/(M + K_M)
'''

t0 = time.time()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = R0Model(loop, infected=["I", "V"], dfe_timeout=10)   # 10 s limit per block
print(f"Model built in {time.time() - t0:.1f} s\n")
for warning in caught:
    print("Warning:", warning.message)

Model built in 10.5 s



### How each block was solved

`dfe_blocks` reports, in solving order, the status of each block: `"closed form"`, `"time limit"` (not solved within the limit) or `"no closed form"` (SymPy found no closed form, as for the root of a quintic), and the time spent solving it symbolically.

In [4]:
for block in model.dfe_blocks:
    print(f"{str(block['variables']):<18} {block['status']:<13} {block['seconds']:6.2f} s")

print("\nCompact DFE:", model.dfe_compact)
print("Definitions:", model.dfe_definitions)   # None: no closed form, computed numerically
print("Implicit DFE?", model.dfe_is_implicit)

['T']              closed form     0.00 s
['N', 'F', 'M']    time limit     10.01 s

Compact DFE: {T: s/d_T, I: 0, V: 0, N: N_star, M: M_star, F: F_star}
Definitions: {N_star: None, F_star: None, M_star: None}
Implicit DFE? True


### Compact $\mathcal{R}_0$

Although $N^*$ and $M^*$ have no closed form, $\mathcal{R}_0$ does in terms of them:

$$
\mathcal{R}_0 = \frac{\beta\, p\, s}{d_T\,(\delta + k_N N^*)\,(c + k_M M^*)}.
$$

The number of target cells in the absence of infection is $T^* = s/d_T$. Each infected cell produces virus at rate $p$ for a mean time $1/(\delta + k_N N^*)$, and each virion infects at rate $\beta T^*$ for $1/(c + k_M M^*)$. The baseline innate response lowers $\mathcal{R}_0$ through $N^*$ and $M^*$.

In [5]:
display(Markdown(f"$$\\mathcal{{R}}_0 = {model.latex()}$$"))

$$\mathcal{R}_0 = \frac{\beta p s}{d_{T} \left(M^{*} k_{M} + c\right) \left(N^{*} k_{N} + \delta\right)}$$

The step-by-step report states which values were not found within the limit:

In [6]:
print(model.report())

pyR0compute: next-generation matrix method
Variables:  T, I, V, N, M, F
Infected:   I, V
Parameters: K_F, K_M, K_N, a_F, a_M, a_N, beta, c, d_F, d_M, d_N, d_T, delta, h_F, h_M, h_N, k_M, k_N, p, s (detected automatically)

New-infection terms F_i (automatic):
  I: T*V*beta
  V: 0

Term classification:
  dI/dt  V  -I*delta   (negative term)
  dI/dt  F  T*V*beta   (transfer from uninfected T)
  dI/dt  V  -I*N*k_N   (negative term)
  dV/dt  V  I*p   (transfer from infected I)
  dV/dt  V  -V*c   (negative term)
  dV/dt  V  -M*V*k_M   (negative term)

Transition terms V_i:
  I: I*N*k_N + I*delta
  V: -I*p + M*V*k_M + V*c

Disease-free equilibrium:
  T = s/d_T
  I = 0
  V = 0
  N = N_star
  M = M_star
  F = F_star
where (solved block by block, in this order):
  N_star = (not found within dfe_timeout = 10 s; computed numerically)
  F_star = (not found within dfe_timeout = 10 s; computed numerically)
  M_star = (not found within dfe_timeout = 10 s; computed numerically)

F =
⎡   β⋅s⎤
⎢0  ───⎥


### Numerical values

Illustrative values (not fitted to data). For each parameter set, the implicit block is solved by integrating the uninfected subsystem towards its steady state and refining with Newton. The block function is compiled once, so each evaluation takes milliseconds.

In [7]:
values = dict(s=10, d_T=0.1, beta=0.005, delta=0.5, k_N=0.2, p=20, c=3, k_M=0.3,
              a_N=1, d_N=0.5, h_N=2, K_F=1, a_M=1, d_M=0.4, h_M=1.5, K_N=1,
              a_F=0.5, d_F=1, h_F=2, K_M=1)

print("Numerical DFE:")
for x, value in model.dfe_numeric(values).items():
    print(f"  {x} = {value:.6f}")
print(f"R0 = {model.R0_numeric(values):.6f}")

t0 = time.time()
for _ in range(100):
    model.R0_numeric(values)
print(f"100 evaluations of R0 in {time.time() - t0:.2f} s")

stab = model.dfe_stability(values)
print("Condition (A5): eigenvalues of the uninfected block =",
      ", ".join(f"{z.real:.4f}" for z in stab["eigenvalues"]),
      "->", "stable" if stab["stable"] else "unstable")

Numerical DFE:


  T = 100.000000
  I = 0.000000
  V = 0.000000
  N = 5.314824
  M = 5.656159
  F = 2.199526
R0 = 1.362211


100 evaluations of R0 in 0.28 s
Condition (A5): eigenvalues of the uninfected block = -0.9985, -0.5082, -0.3932, -0.1000 -> stable


### Threshold

With (A5), the DFE of the full system is stable if $\mathcal{R}_0 < 1$ and unstable if $\mathcal{R}_0 > 1$. We check this with the largest real part of the eigenvalues of the full ($6\times6$) Jacobian at the DFE, varying the infection rate $\beta$.

In [8]:
f = sp.Matrix([model.equations[v] for v in model.variables])
J = f.jacobian(list(model.variables))

for beta in [0.002, 0.004, 0.005, 0.01]:
    v = dict(values, beta=beta)
    Jn = J.subs(model.dfe_numeric(v)).subs({s: v[s.name] for s in J.free_symbols - set(model.variables)})
    abscissa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"beta = {beta:<6} R0 = {model.R0_numeric(v):7.4f}   max Re(λ) = {abscissa:+.4f}")

beta = 0.002  R0 =  0.5449   max Re(λ) = -0.1000
beta = 0.004  R0 =  1.0898   max Re(λ) = +0.1036
beta = 0.005  R0 =  1.3622   max Re(λ) = +0.3993
beta = 0.01   R0 =  2.7244   max Re(λ) = +1.6088


### Effect of the innate response

Since the DFE depends on the parameters of the loop, so does $\mathcal{R}_0$. For example, increasing the activation of NK cells by interferon ($h_N$) raises $N^*$ and, through the loop, $M^*$, and $\mathcal{R}_0$ decreases.

In [9]:
print(f"{'h_N':>5} {'N*':>9} {'M*':>9} {'R0':>9}")
for h_N in [0, 1, 2, 4, 8]:
    v = dict(values, h_N=h_N)
    dfe = {x.name: val for x, val in model.dfe_numeric(v).items()}
    print(f"{h_N:>5} {dfe['N']:9.4f} {dfe['M']:9.4f} {model.R0_numeric(v):9.4f}")

  h_N        N*        M*        R0
    0    2.0000    5.0000    2.4691
    1    3.6548    5.4444    1.7533
    2    5.3148    5.6562    1.3622
    4    8.6389    5.8610    0.9434
    8   15.2912    6.0198    0.5848


With $h_N = 0$ the loop does not act on NK cells and $N^* = a_N/d_N = 2$. With these values, from $h_N = 4$ on the baseline innate response is enough for $\mathcal{R}_0 < 1$: the infection cannot establish itself.

### Global sensitivity

Global sensitivity works as with a closed-form DFE; here $\mathcal{R}_0$ is computed numerically for each sample. Since there is no complete symbolic expression of $\mathcal{R}_0$, the monotonicity required by PRCC is not checked with derivatives (`monotonic` is `None`).

In [10]:
t0 = time.time()
res = model.prcc(baseline=values, spread=0.2, n=1000, seed=1)   # all uniform within ±20 %
print(f"LHS-PRCC (n = 1000) in {time.time() - t0:.1f} s; the 8 most influential parameters:")
table = res.as_dict()
for name in res.ranking()[:8]:
    print(f"  {name:<6} PRCC = {table[name]['PRCC']:+.3f}")

LHS-PRCC (n = 1000) in 5.4 s; the 8 most influential parameters:
  beta   PRCC = +0.887
  s      PRCC = +0.886
  p      PRCC = +0.880
  d_T    PRCC = -0.879
  d_N    PRCC = +0.797
  k_N    PRCC = -0.787
  c      PRCC = -0.762
  h_N    PRCC = -0.643


## 2. Choosing `dfe_timeout`

* **Default (20 s).** Blocks that SymPy solves in closed form take from milliseconds to a few seconds. For example, the coupled block $\{H, R\}$ of the 7-equation model of notebook 03 takes about 6 s (see below) and stays in closed form. On a slower computer such a block could exceed the limit; nothing is lost in that case, it is just computed numerically, and increasing `dfe_timeout` recovers the closed form.
* **A smaller value** (for example 5 s) speeds up the construction of models that you already know have no useful symbolic solution.
* **A larger value** gives SymPy more time if you expect a closed form.
* **`dfe_timeout=None`** removes the limit and solves in the same process, as in version 0.2.0. Use it only if you know that the block finishes.

Models with a simple DFE are unchanged. In the SEIR model, for example, all blocks are solved in closed form in milliseconds:

In [11]:
seir = R0Model('''
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
''', infected=["E", "I"])
for block in seir.dfe_blocks:
    print(f"{str(block['variables']):<10} {block['status']:<12} {block['seconds']:.3f} s")
print("R0 =", seir.R0)

['S']      closed form  0.004 s
['R']      closed form  0.002 s
R0 = Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma))


The 7-equation model of notebook 03 is not affected either: its coupled block $\{H, R\}$ is solved in closed form within the limit.

In [12]:
t0 = time.time()
immune = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])
print(f"Model built in {time.time() - t0:.1f} s")
for block in immune.dfe_blocks:
    print(f"{str(block['variables']):<12} {block['status']:<12} {block['seconds']:.2f} s")

Model built in 6.9 s
['E']        closed form  0.00 s
['H', 'R']   closed form  5.61 s
['C']        closed form  0.02 s
['B']        closed form  0.02 s
